# 전문가 품질점검 데이터 재학습 — ConvNeXt-Tiny

이 노트북은 **자격증명이 없는 Colab ZIP**만 사용합니다. PostgreSQL 접속은 사내 PC의 `prepare_expert_quality_colab_bundle.ps1`에서 `SET TRANSACTION READ ONLY`로 끝내고, Colab에는 DB 비밀번호·DSN을 올리지 않습니다.

Colab 메뉴에서 **런타임 → 런타임 유형 변경 → T4 GPU 이상**을 선택한 후 위에서 아래로 실행하세요.

In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')

BASE = Path('/content/drive/MyDrive/Apartment_Defect_AI')
BUNDLE_NAME = 'expert-quality-convnext-colab.zip'  # PC에서 만든 ZIP 이름과 같게 설정
BUNDLE = BASE / 'colab-input' / BUNDLE_NAME
RESULTS = BASE / 'colab-results'
assert BUNDLE.is_file(), f'Colab ZIP을 찾을 수 없습니다: {BUNDLE}'
assert BUNDLE.with_suffix(BUNDLE.suffix + '.sha256').is_file(), 'ZIP과 함께 .sha256 파일도 업로드하세요.'
RESULTS.mkdir(parents=True, exist_ok=True)
print('입력 ZIP:', BUNDLE)
print('결과 경로:', RESULTS)

In [ ]:
import hashlib, shutil, subprocess, sys, torch
WORK = Path('/content/apartment-defect-expert')
if WORK.exists(): shutil.rmtree(WORK)
WORK.mkdir(parents=True)
shutil.unpack_archive(BUNDLE, WORK)

expected = BUNDLE.with_suffix(BUNDLE.suffix + '.sha256').read_text(encoding='ascii').split()[0].lower()
actual = hashlib.sha256(BUNDLE.read_bytes()).hexdigest()
assert expected == actual, 'ZIP SHA-256 검증 실패: 업로드 파일을 다시 확인하세요.'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{WORK}/project[pytorch]'], check=True)
assert torch.cuda.is_available(), 'GPU가 활성화되지 않았습니다. Colab 런타임을 T4 GPU 이상으로 바꾸세요.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
DATA = Path('/content/expert-quality-data')
if DATA.exists(): shutil.rmtree(DATA)
subprocess.run(['apartment-data', 'vision-colab-prepare', str(WORK), str(DATA), '--workers', '24'], check=True)
import json
manifest = json.loads((DATA / 'colab_dataset_manifest.json').read_text())
print(json.dumps(manifest, ensure_ascii=False, indent=2))
failure_rate = manifest['failure_count'] / max(1, sum(manifest['split_counts'].values()) + manifest['failure_count'])
assert failure_rate <= 0.02, f'이미지 다운로드 실패율 {failure_rate:.2%}가 2%를 초과했습니다. URL/권한을 점검하세요.'

In [ ]:
RUN = Path('/content/expert-quality-convnext-run')
if RUN.exists(): shutil.rmtree(RUN)
subprocess.run([
    'apartment-data', 'vision-train', str(DATA / 'training_spec.json'), str(RUN),
    '--backend', 'pytorch', '--device', 'cuda',
    '--architecture', 'convnext_tiny', '--pretrained',
    '--epochs', '12', '--batch-size', '32', '--learning-rate', '0.0001'
], check=True)
metrics = json.loads((RUN / 'final_metrics.json').read_text())
print(json.dumps(metrics, ensure_ascii=False, indent=2))

## 결과 판정

다음 수치를 확인합니다. `part_detail`과 `cause`는 특히 **Macro-F1**, 클래스별 표본 수, Top-3를 함께 봅니다. Top-3가 높아도 원인 자동확정은 하지 않고 점검자 후보 선택으로 사용합니다.

- 새로운 단지/시점으로 분리한 외부 테스트가 없는 경우, 이 결과는 내부 검증값입니다.
- 하자 원인과 공종은 사진 단독보다 사진 + 점검자 의견 + 계측값 + 유사사례를 결합해 운영합니다.
- 기존 모델보다 외부 검증과 점검자 수정률이 개선된 경우에만 배포 후보로 승인합니다.

In [ ]:
import datetime
stamp = datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
archive_base = RESULTS / f'expert-quality-convnext-run-{stamp}'
zip_path = Path(shutil.make_archive(str(archive_base), 'zip', RUN.parent, RUN.name))
digest = hashlib.sha256()
with zip_path.open('rb') as stream:
    for block in iter(lambda: stream.read(1024 * 1024), b''):
        digest.update(block)
zip_path.with_suffix('.zip.sha256').write_text(f'{digest.hexdigest()}  {zip_path.name}\n', encoding='ascii')
print('결과 ZIP:', zip_path)
print('SHA-256:', digest.hexdigest())
print('PC에서 결과 가져오기: apartment-data vision-colab-import <ZIP> workspace/datasets/expert-quality-colab-run')